# Session 3 — Feature Engineering and Preprocessing

## tl;dr

This notebook creates a reproducible, stratified 80/20 split and a leakage-safe preprocessing pipeline for the Telco churn dataset. It removes the non-predictive `customerID`, maps `Churn` to 0/1, adds `tenure_group` and `num_services`, imputes and scales numeric features, and one-hot encodes categorical features. The result is a model-ready training matrix with 5,634 rows. Model training is intentionally outside this session.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the ChurnSense project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    SERVICE_COLUMNS,
    build_preprocessor,
    engineer_features,
    get_feature_names,
    load_telco_data,
    prepare_features_target,
    split_data,
)

DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20

## Context & Methods

The source is the IBM Telco Customer Churn CSV stored at `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`. One row represents one customer, and `Churn` indicates whether that customer left.

Method and assumptions:

- `customerID` is dropped because it is a unique identifier, not a repeatable customer behavior. Retaining it would add high-cardinality noise and could encourage memorization.
- `Churn` is encoded as `Yes = 1` and `No = 0`.
- Categorical variables use one-hot encoding because their labels have no natural order. Integer/label encoding could incorrectly imply that one plan or payment method is greater than another.
- Numeric variables are median-imputed and standardized so later scale-sensitive models can compare coefficients and optimize reliably. The 11 blank `TotalCharges` values remain missing until the training-only imputer is fitted.
- The split is stratified to preserve the churn rate in both partitions.
- All learned transformations are fitted only on the training partition, then applied unchanged to the test partition.
- Class imbalance is not altered here. `class_weight="balanced"` preserves the observed training rows, while SMOTE creates synthetic minority examples; that choice belongs to Session 4 model evaluation and must only use training data.

## Data

In [2]:
raw = load_telco_data(DATA_PATH)

print(f"Source: {DATA_PATH}")
print(f"Raw shape: {raw.shape}")
print(f"Duplicate rows: {raw.duplicated().sum()}")
display(raw.head(3))
display(
    raw["Churn"]
    .value_counts(dropna=False)
    .rename_axis("Churn")
    .to_frame("customers")
    .assign(rate=lambda table: table["customers"] / len(raw))
)

Source: C:\Users\hp\Desktop\churnsense\data\WA_Fn-UseC_-Telco-Customer-Churn.csv
Raw shape: (7043, 21)
Duplicate rows: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


,customers,rate
Churn,,
No,5174,0.73463
Yes,1869,0.26537


## Results

### Feature engineering and target preparation

`tenure_group` turns tenure into interpretable lifecycle bands. `num_services` counts active phone, line, security, backup, protection, support, and streaming services, providing one compact measure of product adoption.

In [3]:
engineered = engineer_features(raw)
X, y = prepare_features_target(raw)

preparation_summary = pd.Series(
    {
        "input_rows": len(raw),
        "predictor_columns": X.shape[1],
        "encoded_churn_rate": y.mean(),
        "TotalCharges_missing_for_imputation": X["TotalCharges"].isna().sum(),
        "customerID_removed": "customerID" not in X.columns,
        "target_removed_from_X": "Churn" not in X.columns,
    },
    name="value",
)
display(preparation_summary.to_frame())

assert X.shape == (7043, 21)
assert set(y.unique()) == {0, 1}
assert "customerID" not in X and "Churn" not in X
assert {"tenure_group", "num_services"}.issubset(X.columns)

,value
input_rows,7043
predictor_columns,21
encoded_churn_rate,0.26537
TotalCharges_missing_for_imputation,11
customerID_removed,True
target_removed_from_X,True


In [4]:
feature_example_columns = [
    "tenure",
    "tenure_group",
    *SERVICE_COLUMNS,
    "num_services",
]
display(engineered.loc[:, feature_example_columns].head(8))

display(
    pd.DataFrame(
        {
            "tenure_group": X["tenure_group"].value_counts(sort=False),
            "churn_rate": y.groupby(X["tenure_group"], observed=True).mean(),
        }
    ).sort_index()
)

,tenure,tenure_group,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,num_services
0,1,0-12 months,No,No phone service,No,Yes,No,No,No,No,1
1,34,25-48 months,Yes,No,Yes,No,Yes,No,No,No,3
2,2,0-12 months,Yes,No,Yes,Yes,No,No,No,No,3
3,45,25-48 months,No,No phone service,Yes,No,Yes,Yes,No,No,3
4,2,0-12 months,Yes,No,No,No,No,No,No,No,1
5,8,0-12 months,Yes,Yes,No,No,Yes,No,Yes,Yes,5
6,22,13-24 months,Yes,Yes,No,Yes,No,No,Yes,No,4
7,10,0-12 months,No,No phone service,Yes,No,No,No,No,No,1


,tenure_group,churn_rate
tenure_group,,
0-12 months,2186,0.474382
13-24 months,1024,0.287109
25-48 months,1594,0.203890
49-60 months,832,0.144231
61+ months,1407,0.066098


### Stratified split and leakage-safe transformation

The raw feature frame is split first. The `ColumnTransformer` is then fitted with `X_train` only: numeric medians, scaling parameters, and one-hot categories therefore cannot learn from the held-out rows.

In [5]:
X_train, X_test, y_train, y_test = split_data(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_test)],
        "raw_features": [X_train.shape[1], X_test.shape[1]],
        "churn_rate": [y_train.mean(), y_test.mean()],
    },
    index=["train", "test"],
)
display(split_summary)

assert X_train.shape == (5634, 21)
assert X_test.shape == (1409, 21)
assert X_train.index.intersection(X_test.index).empty
assert abs(y_train.mean() - y_test.mean()) < 0.001

,rows,raw_features,churn_rate
train,5634,21,0.265353
test,1409,21,0.265436


In [6]:
preprocessor = build_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = get_feature_names(preprocessor)

processed_summary = pd.DataFrame(
    {
        "rows": [X_train_processed.shape[0], X_test_processed.shape[0]],
        "model_features": [X_train_processed.shape[1], X_test_processed.shape[1]],
        "missing_values": [
            np.isnan(X_train_processed).sum(),
            np.isnan(X_test_processed).sum(),
        ],
    },
    index=["X_train", "X_test"],
)
display(processed_summary)
print(f"X_train shape: {X_train_processed.shape}")
print(f"X_test shape:  {X_test_processed.shape}")

assert X_train_processed.shape == (5634, len(feature_names))
assert X_test_processed.shape == (1409, len(feature_names))
assert len(feature_names) == len(set(feature_names))
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()

,rows,model_features,missing_values
X_train,5634,52,0
X_test,1409,52,0


X_train shape: (5634, 52)
X_test shape:  (1409, 52)


In [7]:
transformed_preview = pd.DataFrame(
    X_train_processed[:5],
    columns=feature_names,
    index=X_train.index[:5],
)
print(f"Numeric inputs ({len(NUMERIC_FEATURES)}): {list(NUMERIC_FEATURES)}")
print(f"Categorical inputs ({len(CATEGORICAL_FEATURES)}): {list(CATEGORICAL_FEATURES)}")
print(f"First 12 of {len(feature_names)} transformed features:")
display(transformed_preview.iloc[:, :12])

Numeric inputs (4): ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_services']
Categorical inputs (17): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']
First 12 of 52 transformed features:


,tenure,MonthlyCharges,TotalCharges,num_services,gender_Female,gender_Male,SeniorCitizen_No,SeniorCitizen_Yes,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes
3738,0.102371,-0.521976,-0.263290,-0.184954,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0
3151,-0.711743,0.337478,-0.504815,-0.667823,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
4860,-0.793155,-0.809013,-0.751214,-0.184954,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
3867,-0.263980,0.284384,-0.173700,0.780784,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
3810,-1.281624,-0.676279,-0.990851,-1.150692,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0


In [8]:
numeric_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
total_charges_position = list(NUMERIC_FEATURES).index("TotalCharges")
learned_total_charges_median = numeric_imputer.statistics_[total_charges_position]
training_total_charges_median = X_train["TotalCharges"].median()

checks = pd.Series(
    {
        "train_rows_equal_5634": len(X_train) == 5634,
        "test_rows_equal_1409": len(X_test) == 1409,
        "split_indices_are_disjoint": X_train.index.intersection(X_test.index).empty,
        "identifier_and_target_excluded": {"customerID", "Churn"}.isdisjoint(X.columns),
        "engineered_features_present": {"tenure_group", "num_services"}.issubset(X.columns),
        "imputer_learned_training_median": np.isclose(
            learned_total_charges_median, training_total_charges_median
        ),
        "processed_matrices_are_finite": (
            np.isfinite(X_train_processed).all() and np.isfinite(X_test_processed).all()
        ),
    },
    name="passed",
)
display(checks.to_frame())
assert checks.all()

,passed
train_rows_equal_5634,True
test_rows_equal_1409,True
split_indices_are_disjoint,True
identifier_and_target_excluded,True
engineered_features_present,True
imputer_learned_training_median,True
processed_matrices_are_finite,True


## Takeaways

- The stratified split contains **5,634 training rows** and **1,409 test rows**, each with 21 pre-transformation predictors.
- `tenure_group` and `num_services` satisfy the Session 3 feature-engineering requirement and retain business meaning.
- Median imputation, standardization, and one-hot encoding produce finite, consistently ordered model features for both partitions.
- The identifier and target are excluded from the predictors, and every learned preprocessing statistic comes from the training split.
- Session 3 ends with model-ready matrices. Model choice, imbalance handling, cross-validation, and metric evaluation are deferred to Session 4.